In [0]:
%sql

SELECT
    MAX(order_purchase_timestamp) AS latest_order_timestamp
FROM workspace.ecommerce_de_project.silver_orders;

latest_order_timestamp
2018-10-17T17:30:18.000Z


In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.bronze_orders
WHERE order_purchase_timestamp >
(
    SELECT MAX(order_purchase_timestamp)
    FROM workspace.ecommerce_de_project.silver_orders
);

order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


## Incremental Load-এর জন্য Watermark Table

In [0]:
%sql

CREATE TABLE IF NOT EXISTS workspace.ecommerce_de_project.etl_watermark (
    pipeline_name STRING,
    last_processed_timestamp TIMESTAMP,
    updated_at TIMESTAMP
);

## Initial Watermark Set

In [0]:
%sql

INSERT INTO workspace.ecommerce_de_project.etl_watermark
SELECT
    'orders' AS pipeline_name,
    MAX(order_purchase_timestamp) AS last_processed_timestamp,
    CURRENT_TIMESTAMP() AS updated_at
FROM workspace.ecommerce_de_project.silver_orders;

num_affected_rows,num_inserted_rows
1,1


## Watermark verify করো

In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.etl_watermark;

pipeline_name,last_processed_timestamp,updated_at
orders,2018-10-17T17:30:18.000Z,2026-11-25T05:59:57.802Z


## Incremental Load Validation

In [0]:
%sql
SELECT COUNT(*) AS new_orders
FROM workspace.ecommerce_de_project.bronze_orders
WHERE order_purchase_timestamp >
(
   SELECT MAX(last_processed_timestamp)
FROM workspace.ecommerce_de_project.etl_watermark
WHERE pipeline_name = 'orders'
);

new_orders
0


## Watermark Verification

In [0]:
%sql
SELECT
    pipeline_name,
    last_processed_timestamp,
    updated_at
FROM workspace.ecommerce_de_project.etl_watermark
WHERE pipeline_name = 'orders';

pipeline_name,last_processed_timestamp,updated_at
orders,2018-10-17T17:30:18.000Z,2026-11-25T05:59:57.802Z


In [0]:
%sql
SELECT
    pipeline_name,
    run_id,
    status,
    start_time,
    end_time,
    duration_seconds
FROM workspace.ecommerce_de_project.pipeline_run_log
ORDER BY start_time DESC;

pipeline_name,run_id,status,start_time,end_time,duration_seconds
Olist E-Commerce Data Engineering,20261125_053823,SUCCESS,2026-11-25T05:38:23.482Z,2026-11-25T05:38:47.256Z,23.774446
Olist E-Commerce Data Engineering,20261125_045616,SUCCESS,2026-11-25T04:56:16.341Z,2026-11-25T05:12:11.827Z,955.486189


## Pipeline Health Summary

In [0]:
%sql
SELECT
    COUNT(*) AS total_runs,
    SUM(CASE WHEN status = 'SUCCESS' THEN 1 ELSE 0 END) AS successful_runs,
    SUM(CASE WHEN status = 'FAILED' THEN 1 ELSE 0 END) AS failed_runs,
    ROUND(AVG(duration_seconds), 2) AS avg_duration_seconds
FROM workspace.ecommerce_de_project.pipeline_run_log;

total_runs,successful_runs,failed_runs,avg_duration_seconds
2,2,0,489.63


## Data Quality Summary

In [0]:
%sql
SHOW TABLES IN workspace.ecommerce_de_project;

database,tableName,isTemporary
ecommerce_de_project,bronze_order_items,false
ecommerce_de_project,bronze_orders,false
ecommerce_de_project,data_quality_monitoring,false
ecommerce_de_project,data_quality_status,false
ecommerce_de_project,etl_watermark,false
ecommerce_de_project,gold_monthly_business_kpis,false
ecommerce_de_project,gold_monthly_freight,false
ecommerce_de_project,gold_monthly_orders,false
ecommerce_de_project,gold_monthly_revenue,false
ecommerce_de_project,gold_order_status,false


In [0]:
%sql
SELECT *
FROM workspace.ecommerce_de_project.pipeline_health_summary;

total_checks,passed_checks,failed_checks,pipeline_status,evaluated_at
5,5,0,HEALTHY,2026-11-24T09:55:42.922Z


In [0]:
%sql
SELECT *
FROM workspace.ecommerce_de_project.data_quality_monitoring;

metric_name,metric_value,checked_at
QUARANTINE_TOTAL_RECORDS,8,2026-11-24T09:50:31.357Z
MISSING_CRITICAL_FIELDS,0,2026-11-24T09:50:31.357Z
BRONZE_TOTAL_RECORDS,99441,2026-11-24T09:50:31.357Z
SILVER_TOTAL_RECORDS,99433,2026-11-24T09:50:31.357Z
DUPLICATE_ORDER_IDS,0,2026-11-24T09:50:31.357Z


## Data Quality Status

In [0]:
%sql
SELECT *
FROM workspace.ecommerce_de_project.data_quality_status;

metric_name,metric_value,quality_status,evaluated_at
BRONZE_TOTAL_RECORDS,99441,PASS,2026-11-24T09:53:21.211Z
SILVER_TOTAL_RECORDS,99433,PASS,2026-11-24T09:53:21.211Z
QUARANTINE_TOTAL_RECORDS,8,PASS,2026-11-24T09:53:21.211Z
MISSING_CRITICAL_FIELDS,0,PASS,2026-11-24T09:53:21.211Z
DUPLICATE_ORDER_IDS,0,PASS,2026-11-24T09:53:21.211Z


## Silver Order Items Duplicate Validation

In [0]:
%sql
SELECT
    order_id,
    order_item_id,
    COUNT(*) AS duplicate_count
FROM workspace.ecommerce_de_project.silver_order_items
GROUP BY order_id, order_item_id
HAVING COUNT(*) > 1;

order_id,order_item_id,duplicate_count


In [0]:
%sql
SELECT
    (SELECT COUNT(*) FROM workspace.ecommerce_de_project.bronze_order_items) AS bronze_count,
    (SELECT COUNT(*) FROM workspace.ecommerce_de_project.silver_order_items) AS silver_count,
    (SELECT COUNT(*) FROM workspace.ecommerce_de_project.quarantine_order_items) AS quarantine_count,
    (
        (SELECT COUNT(*) FROM workspace.ecommerce_de_project.bronze_order_items)
        -
        (SELECT COUNT(*) FROM workspace.ecommerce_de_project.silver_order_items)
        -
        (SELECT COUNT(*) FROM workspace.ecommerce_de_project.quarantine_order_items)
    ) AS unreconciled_records;

bronze_count,silver_count,quarantine_count,unreconciled_records
112650,112650,0,0


In [0]:

%sql

SELECT *
FROM workspace.ecommerce_de_project.etl_watermark;


pipeline_name,last_processed_timestamp,updated_at
orders,2018-10-17T17:30:18.000Z,2026-11-25T05:59:57.802Z


## Bronze থেকে নতুন Order শনাক্ত করা

In [0]:
%sql

SELECT COUNT(*) AS new_orders_count
FROM workspace.ecommerce_de_project.bronze_orders
WHERE order_purchase_timestamp >
(
    SELECT MAX(last_processed_timestamp)
FROM workspace.ecommerce_de_project.etl_watermark
WHERE pipeline_name = 'orders'
);

new_orders_count
0


In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_de_project.silver_orders;

col_name,data_type,comment
order_id,string,null
customer_id,string,null
order_status,string,null
order_purchase_timestamp,timestamp,null
order_approved_at,timestamp,null
order_delivered_carrier_date,timestamp,null
order_delivered_customer_date,timestamp,null
order_estimated_delivery_date,timestamp,null
# Clustering Information,,
# col_name,data_type,comment


In [0]:
%sql

DESCRIBE TABLE workspace.ecommerce_de_project.bronze_orders;

col_name,data_type,comment
order_id,string,null
customer_id,string,null
order_status,string,null
order_purchase_timestamp,timestamp,null
order_approved_at,timestamp,null
order_delivered_carrier_date,timestamp,null
order_delivered_customer_date,timestamp,null
order_estimated_delivery_date,timestamp,null
# Clustering Information,,
# col_name,data_type,comment


## Incremental MERGE

In [0]:
%sql

MERGE INTO workspace.ecommerce_de_project.silver_orders AS target

USING (
    SELECT *
    FROM workspace.ecommerce_de_project.bronze_orders
    WHERE order_purchase_timestamp >
    (
        SELECT MAX(last_processed_timestamp)
FROM workspace.ecommerce_de_project.etl_watermark
WHERE pipeline_name = 'orders'
    )
) AS source

ON target.order_id = source.order_id

WHEN NOT MATCHED THEN
INSERT *;

num_affected_rows,num_updated_rows,num_deleted_rows,num_inserted_rows
0,0,0,0


## Watermark Update

In [0]:
%sql

UPDATE workspace.ecommerce_de_project.etl_watermark
SET
    last_processed_timestamp = (
        SELECT MAX(order_purchase_timestamp)
        FROM workspace.ecommerce_de_project.silver_orders
    ),
    updated_at = CURRENT_TIMESTAMP()
WHERE pipeline_name = 'orders';

num_affected_rows
1


## Incremental Load verification

In [0]:
%sql

SELECT
    pipeline_name,
    last_processed_timestamp,
    updated_at
FROM workspace.ecommerce_de_project.etl_watermark;

pipeline_name,last_processed_timestamp,updated_at
orders,2018-10-17T17:30:18.000Z,2026-11-25T09:16:01.965Z


In [0]:
%sql

SELECT
    pipeline_name,
    COUNT(*) AS row_count
FROM workspace.ecommerce_de_project.etl_watermark
GROUP BY pipeline_name;

pipeline_name,row_count
orders,3


In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.etl_watermark
WHERE pipeline_name = 'orders'
ORDER BY updated_at DESC;

pipeline_name,last_processed_timestamp,updated_at
orders,2018-10-17T17:30:18.000Z,2026-11-25T09:26:08.310Z
orders,2018-10-17T17:30:18.000Z,2026-11-25T09:25:41.089Z
orders,2018-10-17T17:30:18.000Z,2026-11-25T09:16:01.965Z


In [0]:
%sql

DELETE FROM workspace.ecommerce_de_project.etl_watermark
WHERE pipeline_name = 'orders'
AND updated_at < (
    SELECT MAX(updated_at)
    FROM workspace.ecommerce_de_project.etl_watermark
    WHERE pipeline_name = 'orders'
);

num_affected_rows
2


In [0]:
%sql

SELECT *
FROM workspace.ecommerce_de_project.etl_watermark
WHERE pipeline_name = 'orders';

pipeline_name,last_processed_timestamp,updated_at
orders,2018-10-17T17:30:18.000Z,2026-11-25T09:26:08.310Z
